In [ ]:
from ugot import ugot

got = ugot.UGOT() 
got.initialize("192.168.1.118")
got.load_models(["word_recognition"])

print("Done!")

In [ ]:
try:
    while True:
        # Read the text the robot sees
        text = got.get_words_result()

        # Print it out on the display screen
        got.screen_print_text_newline(text=text, color=1)
        got.play_audio_tts(data=text, voice_type=0, wait=True)

        if text in {"Forward"}:
            got.mecanum_move_speed_times(direction=0, speed=30, times=50, unit=1)
        elif text in {"Left"}:
            got.mecanum_turn_speed_times(turn=2, speed=60, times=90, unit=2)
        elif text in {"Right"}:
            got.mecanum_turn_speed_times(turn=3, speed=60, times=90, unit=2)
        elif text in {"Backward"}:
            got.mecanum_move_speed_times(direction=1, speed=30, times=50, unit=1)

        got.screen_clear()
except KeyboardInterrupt:
    print("Stopped")

In [ ]:
import random
import time

score = 0
round_num = 0
previous_answer = None


def clean_reading(text):
    if not text:
        return ""

    fixes = {
        "O": "0",
        "o": "0",
        "l": "1",
        "I": "1",
        "S": "5",
        "B": "8",
        "b": "6",
        "q": "9",
        "z": "2",
    }

    cleaned = ""
    for ch in str(text):
        ch = fixes.get(ch, ch)
        if ch.isdigit():
            cleaned += ch

    return cleaned


def make_problem():
    op = random.choice(["+", "-", "x"])
    if op == "+":
        a = random.randint(1, 20)
        b = random.randint(1, 20)
        answer = a + b
        spoken = "plus"
    elif op == "-":
        a = random.randint(1, 20)
        b = random.randint(1, a)
        answer = a - b
        spoken = "minus"
    else:
        a = random.randint(1, 12)
        b = random.randint(1, 12)
        answer = a * b
        spoken = "times"
    return a, b, op, spoken, answer


# Main loop
try:
    got.screen_clear()
    got.screen_print_text_newline(text="Math", color=1)
    got.play_audio_tts(
        data="Write your answer and show it to me!", voice_type=0, wait=True
    )

    while True:
        round_num += 1

        a, b, op, spoken, answer = make_problem()
        while answer == previous_answer:
            a, b, op, spoken, answer = make_problem()

        # Show and saty the problem
        got.screen_clear()
        got.screen_print_text_newline(text="Question " + str(round_num), color=1)
        got.screen_print_text_newline(text=f"{a} {op} {b} = ?", color=1)
        got.screen_print_text_newline(text="Score: " + str(score), color=1)
        got.play_audio_tts(data=f"What is {a} {spoken} {b}?", voice_type=0, wait=True)

        guess = ""
        while guess == "" or guess == str(previous_answer):
            guess = clean_reading(got.get_words_result())
            time.sleep(0.5)

        got.screen_print_text_newline(text="You wrote: " + guess, color=1)

        if guess == str(answer):
            score += 1
            got.play_audio_tts(
                data=f"Correct! You have {score} points", voice_type=0, wait=True
            )
        else:
            got.play_audio_tts(
                data=f"You lose social credit. The answer was {answer}.",
                voice_type=0,
                wait=True,
            )

        previous_answer = answer
        time.sleep(2)

except KeyboardInterrupt:
    got.screen_clear()
    got.screen_print_text_newline(text="Final score: " + str(score), color=1)
    print("Stopped. Finaal score:", score)